In [1]:
import pandas as pd

### Data Loading and cleaning

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.shape

(50000, 2)

In [4]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [5]:
df.drop_duplicates(inplace=True)

In [6]:
df.shape

(49582, 2)

### Pre-processing

In [7]:
# Convert to lowercase
# Remove URLS
# Remove HTML tags
# Remove punctuations
# Remove strop word
# Perform stemming 
# Encode Sentiments
# Perform vectorization (TF- IDF)



### 1. Converting to Lowercase 

In [8]:
df["review"] = df["review"].str.lower()

### Removing the URLs

In [9]:
import re

def remove_urls(text):
    text =re.sub(r"http\S+","",text) # (pattern, replace, string) eg- https://www.google.com
    return text
    
df["review"] = df["review"].apply(remove_urls)  

### Remove Punctuations

In [10]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]","",text) # A-Z a-z 0-9 \s  
    return text

df["review"] = df["review"].apply(remove_punctuations)  
    

### 4. Removing HTML tags

In [11]:
def remove_html(text):
    text = re.sub(r"<.*?>","",text) # A-Z a-z 0-9 \s  
    return text

df["review"] = df["review"].apply(remove_html)  

### 5. Removing the Stopwords

In [12]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /Users/taruna/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/taruna/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/taruna/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [14]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")
    
    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")

    return text    


df["review"] = df["review"].apply(remove_stopwords)      

In [15]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6. Stemming

In [16]:
# we are using PorterStemming
from nltk.stem import PorterStemmer

In [17]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)


    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)        

In [18]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 7. Encoding

In [19]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [20]:
y = df["sentiment"]

### 8. Vectorization

In [21]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti br br film techniqu unssum ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1


In [22]:
# Tfidf vectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"] )

In [23]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057169 stored elements and shape (49582, 5000)>

## Dataset & Dataloaders

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [25]:
X_train.shape

(39665, 5000)

In [26]:
X_test.shape

(9917, 5000)

In [27]:
import torch
from torch.utils.data import TensorDataset,DataLoader

In [28]:
X_train = X_train.toarray() # parsed arry to numpy arrray
X_test = X_test.toarray()

In [29]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [30]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

## Build our RNN

In [31]:
import torch.nn as nn
import torch.optim as optim

In [32]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128,num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers =  num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first = True)

        # Fully conected layer
        self.fc = nn.Linear(hidden_size, 1)


    def forward(self, x):
        # optional step => shape(num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size)

        out, _ =self.rnn(x, h0)
        #1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        #2nd values = final hidden state of last timestep

        out = self.fc(out[:, -1,:])
        return out
        

In [33]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the RNN

In [35]:
# unsqueeze squeeze

epochs = 10

for epoch in range(epochs):
    optimizer.zero_grad()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
    
        outputs = model(Xb)
    
        outputs = torch.sigmoid(outputs.squeeze())
        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backpropagation
        optimizer.step() # weight update


    print(f"epoch = {epoch+1}/{epochs} and loss ={loss.item()}")    

epoch = 1/10 and loss =0.20593075454235077
epoch = 2/10 and loss =0.20342615246772766
epoch = 3/10 and loss =0.2601911127567291
epoch = 4/10 and loss =0.14216947555541992
epoch = 5/10 and loss =0.36568373441696167
epoch = 6/10 and loss =0.11480305343866348
epoch = 7/10 and loss =0.19702646136283875
epoch = 8/10 and loss =0.19789820909500122
epoch = 9/10 and loss =0.17766444385051727
epoch = 10/10 and loss =0.22959376871585846


In [36]:
# Evaluate the model

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals *100}")   

accuracy = 85.35847534536654
